In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Paths come from path_config.yaml; edit that file's current_workstation to switch machines.
from path_config import PMOF_CODE_DIR, DATA_BASE_DIR, DATA_ACTIONCLS_DIR

# Add project root to sys.path (so imports like src.data work)
if str(PMOF_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(PMOF_CODE_DIR))

from src import logger

from ultralytics import YOLO

from src.visualization import results_to_frames, save_video, VIZ_PARAMS

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np
import shutil

from src.data import list_record_ids
record_ids = list_record_ids()
from src.visualization import VIZ_PARAMS
bbox_colors = VIZ_PARAMS['gt_bbox_colors']

from src.data import imgid_to_imgpath, imgid_to_annpath, recid_to_annpath, recordid_to_imageids, read_annotation, list_record_ids

In [ ]:
record_ids = list_record_ids()
record_ids

In [ ]:
train_record_ids = ["rec4", "rec22", "rec25", "rec27", "rec28"]
val_record_ids = ["rec29", "rec30"]

In [ ]:
from src.data import imgid_to_imgpath, imgid_to_annpath, recid_to_annpath
from src.data import recordid_to_imageids, read_annotation

# Restructure
Doc for Dataset Structure:  https://docs.ultralytics.com/datasets/classify#folder-structure-example

In [ ]:
#make directories
PMOF_actioncls_base_dir = Path(DATA_ACTIONCLS_DIR)

for split in ["train", "val"]:
    for cls in ["seated", "other"]:
        (PMOF_actioncls_base_dir / split / cls).mkdir(parents=True, exist_ok=True)

In [ ]:
def copy_images_by_action(
    record_ids,
    output_dir,
    split,
    seated_action="seated",
):
    """
    Copy images into:
        output_dir/
        ├── {split}/
        │   ├── seated/
        │   └── other/

    An image is classified as 'seated' only if all annotations
    have action == seated_action. If it contains any other action,
    it is classified as 'other'.

    Returns:
        Number of images processed.
    """
    output_dir = Path(output_dir)

    seated_dir = output_dir / split / "seated"
    other_dir = output_dir / split / "other"

    seated_dir.mkdir(parents=True, exist_ok=True)
    other_dir.mkdir(parents=True, exist_ok=True)

    images_counter = 0

    for rec_id in record_ids:
        image_ids = recordid_to_imageids(rec_id)

        for image_id in image_ids:
            images_counter += 1

            # Get image and annotation paths
            imgpath = Path(imgid_to_imgpath(image_id))
            annpath = imgid_to_annpath(image_id)

            # Sanity check
            if not imgpath.is_file():
                print(f"Warning: image not found: {imgpath}")
                continue

            anns = read_annotation(annpath, image_id)

            # Only person annotations matter for action classification
            person_anns = [
                ann for ann in anns
                if ann.category_name == "person"
            ]

            # Any non-seated action -> "other"
            has_other_action = any(
                box.action != seated_action
                for box in person_anns
            )

            destination_dir = (
                other_dir if has_other_action else seated_dir
            )

            destination = destination_dir / imgpath.name

            shutil.copy2(imgpath, destination)

    print(f"{split}: copied {images_counter} images")

    return images_counter

In [ ]:
copy_images_by_action(val_record_ids, PMOF_actioncls_base_dir, 'val')

In [ ]:
copy_images_by_action(train_record_ids, PMOF_actioncls_base_dir, 'train')

# Train Model

In [ ]:
from ultralytics import YOLO

# Load a model
#model = YOLO("yolo26n-cls.yaml")  # build a new model from YAML
model = YOLO("yolo26s-cls.pt")  # load a pretrained model (recommended for training)
#model = YOLO("yolo26n-cls.yaml").load("yolo26n-cls.pt")  # build from YAML and transfer weights

# Train the model
results = model.train(data=str(PMOF_actioncls_base_dir), epochs=3, imgsz=320)